In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [2]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

scaler = StandardScaler()
X_norm = scaler.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=100)

In [ ]:
from scipy.stats import entropy

class DecisionTreeNode:
    """
        ## Attributes:
            lc   (DecisionTreeNode):  left child
            rc   (DecisionTreeNode):  right child
            
            X    (numpy.ndarray):     training examples at current node
            y    (numpy.ndarray):     training lables at current node
            size ()
            
            split_feat (int):               number of feature to split
            split_thrs (float):             threshold to split on
            
            ig   (float):             information gain
            gini (float):             gini impurity
            split_method (string): defaults to ig
            
            depth    (int):               depth
            
        ## Methods:
            find_split(): Returns and sets the feature and threshold pair that results in the best split
            split(): Returns and sets the left child node and right child node after finding the best split
    """
    
    def __init__(self, X_train, y_train):
        if X_train.shape[0] != y_train.shape[0]:
            raise ValueError("X_train and y_train must have the same number of rows")
        self.X = X_train
        self.y = y_train
        self.size = X_train.shape[0]
    
        self.lc = None
        self.rc = None
        
        self.gini = -1
        self.ig = -1
    
    def __calc_gini(self, p):
        return 1 - np.sum(p ** 2)
    
    def depth(self) -> int:
        """
        Calculates and returns the maximum depth of the tree rooted at this node.
        
        Returns:
            int: The maximum depth of the tree (0 for leaf nodes)
        """
        if self.lc is None and self.rc is None:
            # Leaf node - depth is 0
            return 0
        
        left_depth = self.lc.depth() if self.lc is not None else 0
        right_depth = self.rc.depth() if self.rc is not None else 0
        
        return 1 + max(left_depth, right_depth)

    def count_nodes(self) -> int:
        """
        Counts and returns the total number of nodes in the tree rooted at this node.
        
        Returns:
            int: Total number of nodes in the subtree
        """
        # Count this node
        count = 1
        
        # Add nodes from left subtree
        if self.lc is not None:
            count += self.lc.count_nodes()
        
        # Add nodes from right subtree
        if self.rc is not None:
            count += self.rc.count_nodes()
        
        return count
        
    def find_split(self, split_method: str="ig", update_split_criteria: bool=True) -> tuple[int, float]:
        # Determines and sets the feature and threshold pair that results in the best split
        if not X or not y:
            raise ValueError("Cannot split node with no data X, y")
        
        # partitions the data to two subsets based on the best split
        m, n = X.shape
        
        e = entropy(y)
        
        best_gini = -1
        best_ig = -1
        best_feat = 0
        best_thrs = 0
        
        for j in range(n):
            # sort the data with respect to the jth feature
            sorted_indices = np.argsort(X[:,j])
            sorted_labels = y[sorted_indices]
            for i in range(m-1):
                # determine the threshold to split data on
                t = np.mean(X[i, j], X[i+1, j])
                
                y_left = y[X[j] <= t]
                y_right = y[X[j] > t]
                
                # calculate the split criteria
                if split_method.lower() == "gini":
                    gini = self.__calc_gini(np.array([y_left, y_right]))
                    if gini > best_gini:
                        best_gini = gini
                        best_feat = j
                        best_thrs = t
                elif split_method.lower() in ["ig", "information gain", "gain"]:
                    ig = e - ((y_left.shape[0] / y.shape[0]) * entropy(y_left) + (y_right.shape[0] / y.shape[0]) * entropy(y_right))
                    if ig > best_ig:
                        best_ig = ig
                        best_feat = j
                        best_thrs = t
                else:
                    raise ValueError("Invalid split method")
            
        if update_split_criteria:
            self.ig = best_ig
            self.gini = best_gini
            
        self.split_feat = best_feat
        self.split_thrs = best_thrs
        
        return (self.split_feat, self.split_thrs)
    
    def split(self) -> tuple[DecisionTreeNode, DecisionTreeNode]:
        # Partitions the node according to the best split recursively until a stopping condition is reached
        
        if not self.split_feat or not self.split_thrs:
            self.find_split()     
        
        X_left = X[X[self.split_feat] <= self.split_thrs]
        y_left = y[X[self.split_feat] <= self.split_thrs]
        X_right = X[X[self.split_feat] > self.split_thrs]
        y_right = y[X[self.split_feat] > self.split_thrs]
        
        self.lc = DecisionTreeNode(X_left, y_left)
        self.rc = DecisionTreeNode(X_right, y_right)
        
        return (self.lc, self.rc)

    
class DecisionTreeClassifier:
    """
        ## Attributes:
            root     (DecisionTreeNode): decision tree root node 
            
            X_train  (numpy.ndarray):    training examples
            y_train  (numpy.ndarray):    training lables 
            X_test   (numpy.ndarray):    test examples
            
            min_leaf_size (int)
            max_depth (int)
            max_nodes (int)
            
        ## Methods:
            fit(): Builds a DT based on training data and hyperparameters after setting their values
            predict(): Navigates the tree to predict the test points classes
    """
    
    def __init__(self, min_leaf_size=-1, max_depth=-1, max_nodes=-1) -> None:
        self.min_leaf_size = min_leaf_size
        self.max_depth = max_depth
        self.max_nodes = max_nodes
        
    def fit(self, X_train, y_train, min_leaf_size=-1, max_depth=-1, max_nodes=-1) -> None:
        # Builds a DT based on training data and hyperparameters after setting their values
        
        # update hyperparameters
        self.min_leaf_size = min_leaf_size
        self.max_depth = max_depth
        self.max_nodes = max_nodes
        
        self.root = DecisionTreeNode(X_train, y_train)
        
        # define a queue of nodes to split
        queue = [self.root]
        while queue:
            current = queue.pop(0)
    
            # check if current node can be split
            check_depth = current.depth() <= max_depth
            check_size = current.size >= min_leaf_size
            check_node_count = current.count_nodes() < max_nodes

            can_split = check_depth and check_size and check_node_count
            
            current.find_split()
            if can_split:
                current.split()
                queue.extend(current.split())
        
        

            
            
        
    
    def predict(self, X_test):
        pass
        # return y_pred
        
        
    